# QUBO encoding & a fair mini-benchmark

A short, runnable walkthrough of the pipeline: generate a seeded TSP instance,
encode it as a QUBO, verify the QUBO ↔ Ising conversion, then compare solvers
under a matched wall-clock budget against the exact optimum.

Outputs are stripped on commit; run top-to-bottom to reproduce.

In [ ]:
import numpy as np

from quantum_classical_benchmark.problems.tsp import generate_euclidean_instance, tour_cost
from quantum_classical_benchmark.encoding.qubo import (
    build_tsp_qubo, qubo_energy, qubo_to_ising, ising_energy,
)
from quantum_classical_benchmark.solvers.exact import held_karp_tsp
from quantum_classical_benchmark.solvers.simulated_annealing import (
    SimulatedAnnealingSolver, SimulatedAnnealingConfig,
)
from quantum_classical_benchmark.solvers.quantum_inspired import (
    QuantumInspiredSolver, QuantumInspiredConfig,
)

## 1. A seeded instance

In [ ]:
inst = generate_euclidean_instance(n_cities=8, seed=42)
print('cities:', inst.n_cities)
print('distance matrix shape:', inst.distance_matrix.shape)

## 2. QUBO encoding

An N-city TSP becomes an N²-variable QUBO (`x_{i,p}` = city i at position p).
A feasible tour must have strictly lower QUBO energy than an infeasible one.

In [ ]:
qubo, offset, meta = build_tsp_qubo(inst.distance_matrix)
print('QUBO shape:', qubo.shape, '| variables:', int(meta['n_variables']))

def route_to_bits(route, n):
    x = np.zeros(n * n)
    for pos, city in enumerate(route):
        x[city * n + pos] = 1.0
    return x

feasible = route_to_bits(range(inst.n_cities), inst.n_cities)
infeasible = np.zeros(inst.n_cities ** 2); infeasible[: inst.n_cities] = 1.0
print('E(feasible)  =', round(qubo_energy(feasible, qubo, offset), 2))
print('E(infeasible)=', round(qubo_energy(infeasible, qubo, offset), 2))

## 3. QUBO ↔ Ising round-trip

QAOA needs an Ising Hamiltonian. The Ising energy of spins `z` must equal the
QUBO energy of `x = (1 - z) / 2` for every assignment.

In [ ]:
h, J, const = qubo_to_ising(qubo, offset)
rng = np.random.default_rng(0)
ok = all(
    np.isclose(ising_energy(1 - 2 * (x := rng.integers(0, 2, qubo.shape[0])), h, J, const),
               qubo_energy(x, qubo, offset))
    for _ in range(50)
)
print('Ising == QUBO on 50 random assignments:', ok)

## 4. Fair comparison vs the exact optimum

Same instance, same 0.2 s budget. OR-Tools is optional (`pip install .[ortools]`).

In [ ]:
budget = 0.2
opt_cost, _ = held_karp_tsp(inst.distance_matrix)

sa = SimulatedAnnealingSolver(SimulatedAnnealingConfig()).solve(inst.distance_matrix, budget, seed=1)
qi = QuantumInspiredSolver(QuantumInspiredConfig()).solve(inst.distance_matrix, budget, seed=1)
rows = [('exact', opt_cost), ('simulated_annealing', sa.cost), ('quantum_inspired', qi.cost)]

try:
    from quantum_classical_benchmark.solvers.ortools_solver import solve_with_or_tools
    rows.insert(1, ('ortools', solve_with_or_tools(inst.distance_matrix, budget).cost))
except ImportError:
    print('(OR-Tools not installed; skipping baseline)')

for name, cost in rows:
    gap = 100 * (cost - opt_cost) / opt_cost
    print(f'{name:20s} cost={cost:8.2f}  gap={gap:5.2f}%')

## Takeaway

On instances this small everything ties at the optimum. Push N higher (12, 15,
20) and run `make repro`: OR-Tools holds ~0% gap while the metaheuristics fall
behind. The honest conclusion is that there is **no quantum advantage here** —
the value is the fair, reproducible methodology.